# Phase 3 - Hyperparameter Optimization

In [ ]:
# Public Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import sys

# Custom Imports
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from evaluate import *
from model import *
# from train import *

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## Load Training and Test Data

### Via np export (Comment out if not used)

In [ ]:
# Define paths for data and models
data_dir = Path("../data/processed")

X_train_path = data_dir / "X_train.npy"
X_test_path = data_dir / "X_test.npy"
y_train_path = data_dir / "y_train.npy"
y_test_path = data_dir / "y_test.npy"

# Load the data
X_train = np.load(X_train_path)
X_test = np.load(X_test_path)
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

# Display the first 5 rows of the training and testing sets
print("Train X Set:")
display(X_train[:5])

print("Train y Set:")
display(y_train[:5])

print("Test X Set:")
display(X_test[:5])

print("Test y Set:")
display(y_test[:5])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("X_train dtype:", X_train.dtype)
print("X_test dtype:", X_test.dtype)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Training labels:")
print(np.unique(y_train, return_counts=True))
print("Testing labels:")
print(np.unique(y_test, return_counts=True))


## Hyperparameter Grid Setup (3.1)

In [ ]:
# Develop functions here (Finsihed function go into the .py files in the src folder)

In [ ]:
# Test code cells here


## Model Tuning & Architecture Testing (3.2)

In [ ]:
# Develop functions here (Finsihed function go into the .py files in the src folder)

In [ ]:
# Test code cells here


## Evalute Model Performance

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize Global Variables for model development
THRESHOLD_PERCENTILE = 95

def calculate_reconstruction_error(model, data):
    """Calculate reconstruction error (MSE) for each sample."""
    reconstructed_data = model.predict(data, verbose=0)
    return np.mean(np.square(data - reconstructed_data), axis=1)

def generate_anomaly_metrics_and_threshold(model, X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE):
    """
    Generate anomaly threshold and evaluation metrics with visualization.
    
    Parameters:
    - model: Trained autoencoder model
    - X_train: Training data (for threshold calibration)
    - X_test: Test data (for evaluation)
    - y_test: Test labels
    - percentile: Percentile for threshold determination
    
    Returns:
    - threshold: Calculated anomaly threshold
    - y_pred: Predicted anomaly labels for test set
    """
    # Calculate reconstruction errors (single pass)
    train_errors = calculate_reconstruction_error(model, X_train)
    test_errors = calculate_reconstruction_error(model, X_test)
    
    # Determine threshold from training data
    threshold = np.percentile(train_errors, percentile)
    print(f"Train set Anomaly Threshold (Percentile {percentile}): {threshold:.4f}\n")
    
    # Predict anomalies for test set
    y_pred = np.where(test_errors > threshold, 1, 0)

    # Visualization 1: Reconstruction Error Distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sns.histplot(train_errors, bins=50, kde=True, label='Training', alpha=0.7)
    plt.axvline(threshold, color='r', linestyle='--', linewidth=2, label=f'Threshold ({percentile}th %ile)')
    plt.title('Reconstruction Error Distribution (Training Data)')
    plt.xlabel('Reconstruction Error')
    plt.ylabel('Frequency')
    plt.legend()
    
    # Visualization 2: Test Set Anomaly Detection
    plt.subplot(1, 2, 2)
    colors = ['green' if pred == 0 else 'red' for pred in y_pred]
    plt.scatter(range(len(test_errors)), test_errors, c=colors, alpha=0.6, s=30)
    plt.axhline(threshold, color='r', linestyle='--', linewidth=2, label='Threshold')
    plt.title('Test Set: Reconstruction Error vs Sample Index')
    plt.xlabel('Sample Index')
    plt.ylabel('Reconstruction Error')
    plt.legend()
    
    # plt.tight_layout()
    plt.show()
    
    # Visualization 2: Test Set Anomaly Detection (with Log Scale)
    plt.subplot(1, 2, 2)
    colors = ['green' if pred == 0 else 'red' for pred in y_pred]
    plt.scatter(range(len(test_errors)), test_errors, c=colors, alpha=0.6, s=30)
    plt.axhline(threshold, color='r', linestyle='--', linewidth=2, label='Threshold')
    plt.yscale('log') # <--- Added to handle extreme error scales cleanly
    plt.title('Test Set: Reconstruction Error vs Sample Index (Log Scale)')
    plt.xlabel('Sample Index')
    plt.ylabel('Reconstruction Error (Log Scale)')
    plt.legend()
    
    plt.show()

    # Evaluation Metrics
    print("="*50)
    print("TEST SET EVALUATION METRICS")
    print("="*50)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


    return threshold, y_pred


In [ ]:
print("Generating anomaly metrics and threshold")
# threshold, y_pred = generate_anomaly_metrics_and_threshold(model, X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE)

#### Example Code

In [ ]:
# Load the model (assuming the model is saved in the 'models' directory)
model_path = Path("../models/autoencoder_model.h5")
if model_path.exists():
    model = keras.models.load_model(model_path)
    print("Model loaded successfully.")
else:
    print("Model file not found. Please ensure the model is trained and saved in the 'models' directory.")
    
# Evaluate the model's anomaly detection performance on the test set
# _, _ = generate_anomaly_metrics_and_threshold(..., X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE)



In [ ]:
# Define the hyperparameter grid
hyperparameter_grid = {
    'learning_rate': [0.001, 0.01, 0.1],
    'batch_size': [32, 64, 128],
    'epochs': [10, 20, 30],
    'activation_function': ['relu', 'sigmoid', 'tanh']
}

# Define the model architecture
def create_model(learning_rate, batch_size, epochs, activation_function):
    # Create the model architecture
    model = Sequential()
    model.add(Dense(64, activation=activation_function, input_shape=(784,)))
    model.add(Dense(32, activation=activation_function))
    model.add(Dense(10, activation='softmax'))
    
    # Compile the model
    model.compile(optimizer=Adam(lr=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model

# Perform grid search
for learning_rate in hyperparameter_grid['learning_rate']:
    for batch_size in hyperparameter_grid['batch_size']:
        for epochs in hyperparameter_grid['epochs']:
            for activation_function in hyperparameter_grid['activation_function']:
                # Create the model
                model = create_model(learning_rate, batch_size, epochs, activation_function)
                
                # Train the model
                model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_test, y_test))
                
                # Evaluate the model
                loss, accuracy = model.evaluate(X_test, y_test)
                print(f'Learning Rate: {learning_rate}, Batch Size: {batch_size}, Epochs: {epochs}, Activation Function: {activation_function}, Loss: {loss}, Accuracy: {accuracy}')